In [ ]:
import pandas as pd

# Load dataset
df = pd.read_excel(r"C:\Users\itzme\Downloads\Ecommerce.xlsx")

# -------------------------------
# 1. Handling Missing Values
# -------------------------------
print("Missing values:\n", df.isnull().sum())

# Fill numeric columns with median
df['Price'] = df['Price'].fillna(df['Price'].median())
df['Quantity'] = df['Quantity'].fillna(df['Quantity'].median())

# Fill categorical columns with mode
df['Category'] = df['Category'].fillna(df['Category'].mode()[0])
df['City'] = df['City'].fillna(df['City'].mode()[0])

# Drop rows if critical fields are missing
df.dropna(subset=['OrderID'], inplace=True)

# -------------------------------
# 2. Removing Duplicates
# -------------------------------
df.drop_duplicates(inplace=True)

# -------------------------------
# 3. Correcting Data Types
# -------------------------------
df['OrderDate'] = pd.to_datetime(df['OrderDate'], errors='coerce')
df['Category'] = df['Category'].astype('category')
df['Brand'] = df['Brand'].astype('category')
df['Platform'] = df['Platform'].astype('category')

# -------------------------------
# 4. Creating Derived Columns
# -------------------------------
# TotalAmount = Price × Quantity
df['TotalAmount'] = df['Price'] * df['Quantity']

# Extract Year and Month from OrderDate
df['Year'] = df['OrderDate'].dt.year
df['Month'] = df['OrderDate'].dt.month

# -------------------------------
# 5. Filtering or Aggregating Data
# -------------------------------
# Filter: High-value orders
high_value_orders = df[df['TotalAmount'] > 500]

# Aggregate: Average rating per category
avg_rating = df.groupby('Category')['Rating'].mean()

# Aggregate: Total sales per city
city_sales = df.groupby('City')['TotalAmount'].sum()

print("High-value orders:\n", high_value_orders.head())
print("Average rating per category:\n", avg_rating)
print("Total sales per city:\n", city_sales)




import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Price distribution
print(df['Price'].describe())
sns.histplot(df['Price'], bins=30, kde=True)
plt.title("Price Distribution")

# Quantity distribution
print(df['Quantity'].describe())
sns.histplot(df['Quantity'], bins=30, kde=True)
plt.title("Quantity Distribution")

# Ratings distribution
print(df['Rating'].describe())
sns.histplot(df['Rating'], bins=10, kde=False)
plt.title("Ratings Distribution")

# Category frequency
print(df['Category'].value_counts())
sns.countplot(x='Category', data=df)
plt.title("Orders by Category")

###Electronics and Fashion are almost neck-and-neck, both around 3,000 orders.
###Computers form a strong secondary category, but not in the same league as the top two.
###Accessories and Wearables are niche, each at about one-third of Computers’ volume.


# Correlation matrix
print(df[['Price','Quantity','TotalAmount','Rating']].corr())

# Scatter plot: Price vs Rating
sns.scatterplot(x='Price', y='Rating', data=df)
plt.title("Price vs Rating")
plt.show()

# Groupby: Average TotalAmount by Category
avg_sales_category = df.groupby('Category')['TotalAmount'].mean()
print(avg_sales_category)



###Ratings are scattered across all price ranges (from low-cost items to those near 20,000).
###No visible trend: expensive products don’t consistently get higher or lower ratings.
###This confirms the correlation result — customer satisfaction (rating) is independent of price.
###Price ↔ TotalAmount (0.73): Strong positive correlation. Makes sense — higher prices drive higher total amounts.
###Quantity ↔ TotalAmount (0.60): Also strongly positive. More units sold → higher totals.
###Price ↔ Quantity (~0.007): Essentially no relationship. Expensive items aren’t necessarily bought in larger or smaller quantities.




# Pivot table: Average TotalAmount by Category and City
pivot = pd.pivot_table(df, values='TotalAmount', index='Category', columns='City', aggfunc='mean')
print(pivot)

# Heatmap of correlations
plt.figure(figsize=(8,6))
sns.heatmap(df[['Price','Quantity','TotalAmount','Rating']].corr(), annot=True, cmap="coolwarm")
plt.title("Correlation Heatmap")
plt.show()

# Pairplot
sns.pairplot(df[['Price','Quantity','TotalAmount','Rating','Category']], hue='Category')
plt.show()



# City-level sales stats
print(df.groupby('City')['TotalAmount'].agg(['mean','median','std']))

# Category-level rating stats
print(df.groupby('Category')['Rating'].agg(['mean','count','min','max']))


###Rating ↔ Everything (~0.00): Ratings show almost no correlation with price, quantity, or total amount.
###Category-Level Insights-Electronics (green): High-price, high-total cluster. They’re revenue heavyweights.
###Fashion (red): Lower price points but higher quantities, explaining why it nearly matches Electronics in total orders.
###Computers (orange): Mid-price, mid-total — steady performers.
###Accessories (blue) & Wearables (purple): Lower price and lower totals, niche segments with limited impact.




# Bar plot: Total Sales by Category
category_sales = df.groupby('Category')['TotalAmount'].sum().sort_values()
category_sales.plot(kind='bar', color='teal')
plt.title("Total Sales by Category")
plt.xlabel("Category")
plt.ylabel("Revenue")
plt.show()


###Electronics: Highest total sales revenue — premium pricing drives this dominance.
###Fashion: Strong second place — fewer dollars per item but high volume makes it competitive.
###Computers: Solid mid-tier contributor.
###Accessories & Wearables: Much smaller revenue share, both trailing far behind.



monthly_sales = df.groupby(['Year','Month'])['TotalAmount'].sum()
monthly_sales.plot(kind='line', marker='o', color='navy')
plt.title("Monthly Sales Trend")
plt.xlabel("Year-Month")
plt.ylabel("Total Sales")
plt.show()



###Sales fluctuate between about 22M and 27M across months in 2024.
###There are noticeable peaks and dips, suggesting seasonality or campaign-driven spikes.
###The overall trend looks relatively stable — no dramatic upward or downward slope, but recurring cycles.
###High months could align with festivals, holidays, or promotional events (e.g., Diwali, Christmas, end-of-season sales).
###Low months may reflect off-season demand or gaps in marketing pushes.
###Since Electronics and Fashion are the top categories, their seasonality likely drives these swings.


    

category_counts = df['Category'].value_counts()
category_counts.plot(kind='pie', autopct='%1.1f%%', startangle=90, cmap='Set3')
plt.title("Order Distribution by Category")
plt.ylabel("")  # Hide y-label
plt.show()



###Electronics & Fashion are your core drivers — they’re both popular and revenue-heavy 
###Computers are a strong secondary pillar, worth nurturing.
###Accessories & Wearables may not scale on their own, but they’re perfect for bundling or upselling alongside the big categories.
    


    
plt.hist(df['Price'], bins=30, color='skyblue', edgecolor='black')
plt.title("Price Distribution")
plt.xlabel("Price")
plt.ylabel("Frequency")
plt.show()



###Cross-category leverage: Fashion thrives on volume at lower prices, 
###while Electronics thrive on premium pricing — together, they balance the revenue model.
###Prices range broadly from 0 to 20,000, with a fairly uniform spread across bins.
###Each bin has similar frequencies (around 300–375), meaning there isn’t a single dominant price point — products are offered across the spectrum.
###No sharp peaks or clustering, which suggests the catalog is intentionally diversified across price ranges.
###Revenue drivers: Since Electronics dominate revenue, their higher prices likely sit in the upper bins, 
###while Fashion’s strength comes from the lower-to-mid bins with higher order counts.


sns.boxplot(x='Category', y='TotalAmount', data=df, palette='Set2')
plt.title("Order Value Distribution by Category")
plt.xlabel("Category")
plt.ylabel("Total Amount")
plt.show()


###Electronics: Highest median and widest spread of order values. They generate large transactions, with some extreme outliers at the top end.
###Electronics dominate in high-value orders — they’re the premium driver.
###Fashion: Lower median than Electronics, but still substantial. Distribution is tighter, meaning Fashion orders are more consistent in value.
###Fashion thrives on consistency — lots of orders, but each at a moderate value.
###Computers: Mid-range median, with moderate spread. Orders vary but don’t reach the extremes of Electronics.
###Computers balance between the two — not as premium as Electronics, not as volume-driven as Fashion.
###Accessories & Wearables: Lowest medians and narrower ranges. Their order values are small and relatively uniform compared to other categories.
###Accessories & Wearables are niche — low-value, low-variance, best positioned as add-ons or bundles.




sns.scatterplot(x='Quantity', y='TotalAmount', hue='Category', data=df, alpha=0.7)
plt.title("Quantity vs Total Amount by Category")
plt.xlabel("Quantity")
plt.ylabel("Total Amount")
plt.show()



###Electronics (green): Even with small quantities (1–2 units), they generate very high total amounts. This reinforces their premium pricing power.
###Electronics = high-value, low-quantity model: Customers buy fewer units, but each sale is big.
###Fashion (red): Lower total amounts per order, but spread across higher quantities. Fashion thrives on volume rather than ticket size.
###Fashion = low-value, high-quantity model: Customers buy more units, but each sale is smaller.
###Computers (orange): Mid-range — not as high-value as Electronics, but stronger than Fashion in per-order totals.
###Computers = balanced: Orders are moderate in both quantity and value.
###Accessories (blue) & Wearables (purple): Clustered at low quantities and low totals. They rarely break into high-value orders.
###Accessories & Wearables = niche add-ons: They don’t scale in either dimension.


    
sns.heatmap(df[['Price','Quantity','TotalAmount','Rating']].corr(), annot=True, cmap='coolwarm')
plt.title("Correlation Heatmap")
plt.show()


###Revenue drivers: Price and quantity are the two levers that directly influence total sales.
###Customer satisfaction: Ratings are disconnected from financial metrics, so improving them requires focusing on product quality, service, and customer experience rather than pricing strategy.
###Category dynamics: Electronics dominate revenue because of high prices, Fashion dominates orders because of high quantities — both models align perfectly with these correlations.
###Electronics: Premium pricing strategy is validated — each unit adds significantly to revenue.
###Fashion: Volume strategy is validated — quantity drives totals even at lower prices.
###Computers: Balanced contribution, with room to scale either by premium positioning or volume.
###Accessories & Wearables: Neither price nor quantity is strong, so they’re best leveraged as upsell/cross-sell items.



fig, axes = plt.subplots(1, 2, figsize=(12,5))

sns.histplot(df['TotalAmount'], bins=30, ax=axes[0], color='purple')
axes[0].set_title("Total Amount Distribution")

sns.boxplot(x='Category', y='TotalAmount', data=df, ax=axes[1], palette='Set3')
axes[1].set_title("Order Value by Category")

plt.tight_layout()
plt.show()



###Histogram
###The purple histogram shows that most orders cluster in the lower-to-mid ranges, with fewer very high-value transactions.
###This confirms that while big-ticket orders exist (especially in Electronics), the bulk of the business comes from smaller, more frequent transactions.

###Boxplot (Order Value by Category)
###Electronics: Highest median and widest spread — they dominate high-value orders.
###Fashion: Lower median but consistent distribution, reflecting its volume-driven model.
###Computers: Mid-range, steady contributor.
###Accessories & Wearables: Lowest medians, tightly clustered — small, predictable order values.
###Revenue mix: Electronics drive spikes in high-value orders, while Fashion ensures steady volume.
###Risk balance: Fashion’s consistency stabilizes revenue, Electronics provide upside potential.
###Strategic role of smaller categories: Accessories and Wearables won’t lead revenue, but they’re reliable and can be leveraged for upselling.



